# Notebook 01 — Python Data Validation

This notebook loads the analysis-ready TCGA-BRCA cohort produced by the R cleaning pipeline (`R/02_clean_tcga_cdr.R`) and performs a basic data validation in Python. The goals are to:

1. Confirm the dataset dimensions and variable types.
2. Quantify missingness across all columns.
3. Inspect the distributions of key variables before any modelling is carried out.

**Input:** `data/processed/tcga_brca_survival_clean.csv`  
**Produced by:** `R/02_clean_tcga_cdr.R`  
**Next step:** `notebooks/02_ml_risk_prediction_demo.ipynb`

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed")

df = pd.read_csv(DATA_DIR / "tcga_brca_survival_clean.csv")

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

## Missingness

Missing data is quantified for every column. The columns with the highest missing rates are:

- **`race`** and **`menopause_status`**: both 7.7 % — these are reported inconsistently in the TCGA clinical data and will be imputed inside the ML pipeline.
- **`stage_group`**: 2.2 % (n = 24) — these patients have `stage_raw = "Stage X"`, a TCGA coding artefact that does not map to any standard AJCC stage group. They are excluded from the Cox regression model and from the ML modelling in notebook 02.
- **`stage_raw`**: 0.5 % (n = 5) — a small number of patients with no stage information at all.

Age, survival outcome, and follow-up time are complete for all 1 083 patients.

In [ ]:
missingness = (
    df.isna()
    .sum()
    .reset_index(name="missing_n")
    .rename(columns={"index": "variable"})
)

missingness["missing_percent"] = round(
    100 * missingness["missing_n"] / len(df), 1
)

missingness = missingness.sort_values("missing_percent", ascending=False)

display(missingness)

# missingness.to_csv(DATA_DIR / "python_missingness_summary.csv", index=False)

## Distribution checks

Univariate distributions are inspected to detect data quality issues before modelling. Key observations:

- **Overall survival events:** 151 of 1 083 patients (14 %) experienced death — a low event rate typical of TCGA-BRCA, which benefits from favourable prognosis and variable follow-up length.
- **Follow-up time:** Median ≈ 28.3 months; maximum ≈ 282.7 months (~23.6 years), consistent with long-term TCGA follow-up in earlier cohorts.
- **Stage distribution:** Stage II is the largest group (n = 612); Stage IV is small (n = 20), which limits the precision of Stage IV estimates throughout the analysis.
- **Race / menopause status:** Small `"[Not Evaluated]"` categories (3 and 6 patients respectively) are retained as nominal levels and handled by one-hot encoding in notebook 02.

In [ ]:
display(df["os_event"].value_counts(dropna=False))
display(df["os_time_months"].describe())
display(df["stage_group"].value_counts(dropna=False))
display(df["race"].value_counts(dropna=False))
display(df["menopause_status"].value_counts(dropna=False))
display(df["histological_type"].value_counts(dropna=False))